In [29]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K
from sklearn.model_selection import train_test_split


In [30]:
base_path = "processed_signature/processed"
csv_path = os.path.join(base_path, "signature_dataset.csv")

df = pd.read_csv(csv_path)

print(df.head())


            image_path  person_id  label split
0  test/049/01_049.png         49      0  test
1  test/049/02_049.png         49      0  test
2  test/049/03_049.png         49      0  test
3  test/049/04_049.png         49      0  test
4  test/049/05_049.png         49      0  test


In [31]:
def load_image(img_path):
    img = image.load_img(os.path.join(base_path, img_path), target_size=(224,224))
    img = image.img_to_array(img)
    img = img / 255.0
    return img


In [32]:
print(df.columns)


Index(['image_path', 'person_id', 'label', 'split'], dtype='str')


In [33]:
#Create Pairs for Siamese Training

def create_pairs(df):
    pairs = []
    labels = []

    grouped = df.groupby("person_id")

    for writer_id, group in grouped:
        genuine = group[group['label'] == 0]['image_path'].values
        forged  = group[group['label'] == 1]['image_path'].values

        # Genuine-Genuine pairs (label=1)
        for i in range(len(genuine)-1):
            img1 = load_image(genuine[i])
            img2 = load_image(genuine[i+1])
            pairs.append([img1, img2])
            labels.append(1)

        # Genuine-Forged pairs (label=0)
        for i in range(min(len(genuine), len(forged))):
            img1 = load_image(genuine[i])
            img2 = load_image(forged[i])
            pairs.append([img1, img2])
            labels.append(0)

    return np.array(pairs), np.array(labels)


In [34]:
pairs, labels = create_pairs(df)


In [35]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    pairs, labels, test_size=0.2, random_state=42
)

# train_df = df[df["split"] == "train"]
# test_df  = df[df["split"] == "test"]


# Hare i changed something

In [36]:
def l2_normalize(x):
    return K.l2_normalize(x, axis=1)

In [37]:
# #Build Embedding Network (ResNet50)
# def build_embedding_model():

#     base_model = ResNet50(
#         weights='imagenet',
#         include_top=False,
#         input_shape=(224,224,3)
#     )

#     base_model.trainable = False
#     # base_model.trainable = True

#     # for layer in base_model.layers[:-30]:
#     #    layer.trainable = False

#     x = base_model.output
#     x = GlobalAveragePooling2D()(x)
#     x = Dense(256, activation='relu')(x)

#     model = Model(base_model.input, x)
#     return model

from tensorflow.keras.applications import MobileNetV2

def build_embedding_model():

    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224,224,3)
    )

    base_model.trainable = True

    # Freeze early layers only
    for layer in base_model.layers[:-40]:
        layer.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)

    x = Lambda(lambda t: K.l2_normalize(t, axis=1))(x)


    return Model(base_model.input, x)

In [38]:
#Build Siamese Network
def euclidean_distance(vectors):
    x, y = vectors
    return K.sqrt(K.sum(K.square(x - y), axis=1, keepdims=True))


# hare also

In [39]:
embedding_model = build_embedding_model()

input_a = Input(shape=(224,224,3))
input_b = Input(shape=(224,224,3))

embedding_a = embedding_model(input_a)
embedding_b = embedding_model(input_b)

# distance = Lambda(euclidean_distance)([embedding_a, embedding_b])

# output = Dense(1, activation='sigmoid')(distance)

# siamese_model = Model(inputs=[input_a, input_b], outputs=output)

# siamese_model.compile(
#     loss='binary_crossentropy',
#     optimizer=Adam(0.0001),
#     metrics=['accuracy']
# )



distance = Lambda(euclidean_distance)([embedding_a, embedding_b])

siamese_model = Model(inputs=[input_a, input_b], outputs=distance)

def contrastive_loss(y_true, y_pred):
    margin = 1
    return K.mean(
        y_true * K.square(y_pred) +
        (1 - y_true) * K.square(K.maximum(margin - y_pred, 0))
    )

siamese_model.compile(
    loss=contrastive_loss,
    optimizer=Adam(1e-4)
)

siamese_model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_5       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_2        │ (None, 128)       │  2,421,952 │ input_layer_4[0]… │
│ (Functional)        │                   │            │ input_layer_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 1)         │          0 │ functional_2[0][… │
│                     │                   │            │ functional_2[1][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,421,952 (9.24 MB)

 Trainable params: 1,845,504 (7.04 MB)

 Non-trainable params: 576,448 (2.20 MB)

In [40]:
#Balance Pairs
print("Positive pairs:", np.sum(labels==1))
print("Negative pairs:", np.sum(labels==0))

Positive pairs: 2085
Negative pairs: 1907


In [41]:
#Train Model
siamese_model.fit(
    [X_train[:,0], X_train[:,1]],
    y_train,
    validation_data=([X_test[:,0], X_test[:,1]], y_test),
    epochs=10,
    batch_size=8
)


Epoch 1/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 243s 533ms/step - loss: 0.2212 - val_loss: 0.2016
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 218s 544ms/step - loss: 0.1684 - val_loss: 0.1802
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 219s 547ms/step - loss: 0.1368 - val_loss: 0.1573
Epoch 4/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 220s 551ms/step - loss: 0.1130 - val_loss: 0.1461
Epoch 5/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 217s 542ms/step - loss: 0.0947 - val_loss: 0.1571
Epoch 6/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 219s 547ms/step - loss: 0.0793 - val_loss: 0.1363
Epoch 7/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 216s 541ms/step - loss: 0.0700 - val_loss: 0.1310
Epoch 8/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 220s 549ms/step - loss: 0.0594 - val_loss: 0.1452
Epoch 9/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 240s 599ms/step - loss: 0.0542 - val_loss: 0.1243
Epoch 10/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 254s 636ms/step - loss: 0.0443 - val_loss: 0.1172


#Check separation

In [42]:
preds = siamese_model.predict([X_test[:,0], X_test[:,1]])

genuine = preds[y_test == 1]
forged  = preds[y_test == 0]

print("Mean Genuine Distance:", np.mean(genuine))
print("Mean Forged Distance:", np.mean(forged))

25/25 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step
Mean Genuine Distance: 0.29696214
Mean Forged Distance: 0.8901422


In [43]:
siamese_model.save("signature_siamese_model.h5")


#Customer Registration

In [44]:
#Customer Registration Function
def get_embedding(img_path):
    img = load_image(img_path)
    img = np.expand_dims(img, axis=0)
    embedding = embedding_model.predict(img)
    return embedding


In [45]:
customer_database = {}


In [46]:
def register_customer(customer_id, signature_paths):
    embeddings = []
    for path in signature_paths:
        emb = get_embedding(path)
        embeddings.append(emb)
    customer_database[customer_id] = embeddings


In [47]:
def verify_signature(customer_id, test_signature_path):

    test_embedding = get_embedding(test_signature_path)

    stored_embeddings = customer_database[customer_id]

    distances = []

    for emb in stored_embeddings:
        dist = np.linalg.norm(test_embedding - emb)
        distances.append(dist)

    avg_distance = np.mean(distances)

    threshold = 0.5  # You can tune this

    if avg_distance < threshold:
        print("Genuine Signature")
        print("Confidence:", round((1-avg_distance)*100,2), "%")
    else:
        print("Forged Signature")
        print("Confidence:", round(avg_distance*100,2), "%")


In [48]:
def get_embedding(img_path):
    img = image.load_img(img_path, target_size=(224,224))
    img = image.img_to_array(img)
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    embedding = embedding_model.predict(img)
    return embedding


In [49]:
import os

def register_customer(customer_id, folder_path):

    embeddings = []

    # Get all image files from folder
    image_files = [
        os.path.join(folder_path, file)
        for file in os.listdir(folder_path)
        if file.lower().endswith(('.png', '.jpg', '.jpeg'))
    ]

    if len(image_files) == 0:
        print("No images found in folder!")
        return

    for img_path in image_files:
        emb = get_embedding(img_path)
        embeddings.append(emb)

    customer_database[customer_id] = embeddings

    print(f"✅ Customer {customer_id} registered successfully.")
    print(f"Stored {len(embeddings)} signature samples.")


In [50]:
register_customer("C101", "processed")


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
✅ Customer C101 registered successfully.
Stored 12 signature samples.


In [51]:
import numpy as np

def verify_signature(customer_id, test_signature_path, threshold=0.6):

    if customer_id not in customer_database:
        print("Customer not found!")
        return

    test_embedding = get_embedding(test_signature_path)

    stored_embeddings = customer_database[customer_id]

    distances = []

    for emb in stored_embeddings:
        dist = np.linalg.norm(test_embedding - emb)
        distances.append(dist)

    avg_distance = np.mean(distances)

    print("Average Distance:", avg_distance)

    if avg_distance < threshold:
        confidence = (1 - avg_distance/threshold) * 100
        print("✅ Genuine Signature")
        print("Confidence:", round(confidence,2), "%")
    else:
        confidence = (avg_distance/threshold - 1) * 100
        print("❌ Forged Signature")
        print("Confidence:", round(confidence,2), "%")


In [54]:
verify_signature(
    "C101",
    "more forg.jpeg"
)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Average Distance: 0.7545436
❌ Forged Signature
Confidence: 25.76 %
